# 06 — Silver Bureau

**Credit Risk Intelligence Platform** — Camada Silver

Este notebook transforma a tabela Bronze `credit_risk.bronze.bureau` em uma tabela Silver tratada, padronizada e preparada para análise e Machine Learning.

## Pipeline

```
credit_risk.bronze.bureau  →  credit_risk.silver.bureau
```

## Sobre a tabela bureau

A tabela `bureau` contém informações de créditos anteriores do cliente obtidas de fontes externas (bureaus de crédito). Cada registro representa um crédito reportado por um bureau externo, identificado por `SK_ID_BUREAU`. O relacionamento com a tabela de aplicação é baseado em `SK_ID_CURR`.

A tabela Silver `bureau` permanece no **nível original dos registros** — nenhuma agregação por cliente é realizada neste notebook.

## Transformações aplicadas

1. **Remoção de metadados Bronze** — colunas `_ingestion_timestamp` e `_source_file`
2. **Padronização de categorias** — `trim()` em colunas string
3. **Preservação de NULLs numéricos** — imputação é responsabilidade do ML prep
4. **Flags de validação** — valores monetários negativos, datas futuras e valores inválidos
5. **Colunas de controle** — timestamp, versão, origem, hash
6. **Auditoria** — registro completo da transformação

## Regras

> A Bronze **NÃO é modificada**. Todas as transformações criam novas tabelas Silver.
> Nenhum registro é removido sem justificativa documentada.
> Nenhuma agregação por cliente é realizada — a tabela permanece no nível de registros originais.

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração, Imports e Parâmetros
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Parâmetros do pipeline
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "silver_v1.0"
NOTEBOOK_NAME = "06_silver_bureau"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"silver_bureau_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)

# ----------------------------------------------------------------------------
# Tabelas de origem (Bronze) e destino (Silver)
# ----------------------------------------------------------------------------
BRONZE_TABLE = "credit_risk.bronze.bureau"
SILVER_TABLE = "credit_risk.silver.bureau"
AUDIT_TABLE = "credit_risk.silver.audit_transformation"

# Tabelas de aplicação Silver (para integridade referencial)
SILVER_APP_TRAIN = "credit_risk.silver.application_train"
SILVER_APP_TEST = "credit_risk.silver.application_test"

# ----------------------------------------------------------------------------
# Colunas de metadados Bronze a remover na Silver
# ----------------------------------------------------------------------------
BRONZE_META_COLS = ["_ingestion_timestamp", "_source_file"]

# ----------------------------------------------------------------------------
# Criar schema Silver se não existir
# ----------------------------------------------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk.silver")
print(f"Schema credit_risk.silver verificado/criado.")

# ----------------------------------------------------------------------------
# Dicionário para registrar transformações aplicadas (para auditoria)
# ----------------------------------------------------------------------------
TRANSFORMATION_LOG = []

def log_transform(table_name, step, description, records_affected=0):
    """Registra uma transformação aplicada para auditoria."""
    TRANSFORMATION_LOG.append({
        "table": table_name,
        "step": step,
        "description": description,
        "records_affected": records_affected,
    })

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline Version: {PIPELINE_VERSION}")

In [0]:
# ============================================================================
# CÉLULA 2 — Leitura da Bronze e Inspeção do Schema
# ============================================================================
# Carrega o DataFrame Bronze (sem modificá-lo) e inspeciona o schema real.

df_bureau_bronze = spark.table(BRONZE_TABLE)

# Métricas básicas
bronze_row_count = df_bureau_bronze.count()
bronze_col_count = len(df_bureau_bronze.columns)

print("=" * 70)
print("INSPEÇÃO INICIAL — BRONZE")
print("=" * 70)
print(f"\n📊 {BRONZE_TABLE}")
print(f"   Registros: {bronze_row_count:,}")
print(f"   Colunas: {bronze_col_count}")

# ----------------------------------------------------------------------------
# Schema detalhado (tipos e nullable)
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print("SCHEMA — bureau (tipos e nullable)")
print("─" * 70)
for field in df_bureau_bronze.schema.fields:
    print(f"   {field.name:<30} {field.dataType.simpleString():<12} nullable={field.nullable}")

# ----------------------------------------------------------------------------
# Verificar colunas-chave esperadas
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print("COLUNAS-CHAVE")
print("─" * 70)
key_cols = ["SK_ID_CURR", "SK_ID_BUREAU"]
for c in key_cols:
    if c in df_bureau_bronze.columns:
        print(f"   ✅ {c}: presente")
    else:
        print(f"   ❌ {c}: AUSENTE")

print("\n✅ Leitura da Bronze concluída!")

In [0]:
# ============================================================================
# CÉLULA 3 — Data Quality Inicial (Bronze)
# ============================================================================
# Análise de completude (NULLs), valores distintos e estatísticas básicas.
# Usado como baseline para comparar Bronze → Silver.

def compute_null_summary(df, table_name, row_count):
    """Computa resumo de NULLs para todas as colunas de um DataFrame."""
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
    null_row = df.agg(*null_exprs).collect()[0]
    null_pairs = [(c, null_row[c]) for c in df.columns if null_row[c] and null_row[c] > 0]
    null_pairs.sort(key=lambda x: x[1], reverse=True)
    return null_pairs

# ----------------------------------------------------------------------------
# NULLs por coluna
# ----------------------------------------------------------------------------
bronze_nulls = compute_null_summary(df_bureau_bronze, BRONZE_TABLE, bronze_row_count)
sep = "─" * 70
print(sep)
print(f"NULLs POR COLUNA — {BRONZE_TABLE} ({len(bronze_nulls)} cols com nulls)")
print(sep)
for c, n in bronze_nulls:
    print(f"   {c:<30} {n:>10,}  ({n/bronze_row_count*100:.2f}%)")

# ----------------------------------------------------------------------------
# Valores distintos por coluna
# ----------------------------------------------------------------------------
print()
print(sep)
print("VALORES DISTINCTOS POR COLUNA")
print(sep)
print()
for c in df_bureau_bronze.columns:
    if c not in BRONZE_META_COLS:
        d = df_bureau_bronze.select(c).distinct().count()
        print(f"   {c:<30} {d:>10,}")

# ----------------------------------------------------------------------------
# Valores categóricos distintos
# ----------------------------------------------------------------------------
cat_cols = [f.name for f in df_bureau_bronze.schema.fields if f.dataType.simpleString() == "string" and f.name not in BRONZE_META_COLS]
print()
print(sep)
print("VALORES CATEGÓRICOS DISTINTOS")
print(sep)
for c in cat_cols:
    vals = df_bureau_bronze.groupBy(c).count().orderBy(F.desc("count")).collect()
    print(f"\n   {c} ({len(vals)} distinct):")
    for r in vals:
        print(f"      {str(r[c]):<45} {r['count']:>12,}")

# ----------------------------------------------------------------------------
# Estatísticas numéricas (min, max, mean)
# ----------------------------------------------------------------------------
numeric_cols = [f.name for f in df_bureau_bronze.schema.fields
                if f.dataType.simpleString() in ("int", "double", "long", "float")
                and f.name not in BRONZE_META_COLS]
print()
print(sep)
print("ESTATÍSTICAS NUMÉRICAS (min, max, mean)")
print(sep)
for c in numeric_cols:
    stats = df_bureau_bronze.select(c).summary("min", "max", "mean").collect()
    median = df_bureau_bronze.approxQuantile(c, [0.5], 0.01)
    med_str = f"{median[0]:.2f}" if median else "N/A"
    print(f"   {c:<30} min={str(stats[0][c]):>15}  max={str(stats[1][c]):>15}  mean={str(round(float(stats[2][c]),2) if stats[2][c] else 'N/A'):>15}  median={med_str}")

print("\n✅ Data Quality inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Perfil da Tabela Bureau e Validação de Identificadores
# ============================================================================
# bureau contém informações de créditos anteriores do cliente obtidas de
# fontes externas (bureaus de crédito). O relacionamento com a tabela de
# aplicação é baseado em SK_ID_CURR. O identificador do registro de bureau
# é SK_ID_BUREAU.

print("=" * 70)
print("PERFIL DA TABELA BUREAU")
print("=" * 70)
print("""
bureau contém informações de créditos anteriores do cliente obtidas de
fontes externas (bureaus de crédito).

Relacionamento esperado:
  - SK_ID_CURR   → chave estrangeira para application_train/test
  - SK_ID_BUREAU → identificador único do registro de bureau

Nenhuma agregação por cliente é realizada neste notebook.
A tabela Silver permanece no nível original dos registros.
""")

# ----------------------------------------------------------------------------
# Validação de identificadores
# ----------------------------------------------------------------------------
print(f"{'─' * 70}")
print("VALIDAÇÃO DE IDENTIFICADORES")
print(f"{'─' * 70}")

# SK_ID_CURR
sk_curr_null = df_bureau_bronze.filter(F.col("SK_ID_CURR").isNull()).count()
sk_curr_distinct = df_bureau_bronze.select("SK_ID_CURR").distinct().count()
sk_curr_min = df_bureau_bronze.select(F.min("SK_ID_CURR")).collect()[0][0]
sk_curr_max = df_bureau_bronze.select(F.max("SK_ID_CURR")).collect()[0][0]
print(f"\n   SK_ID_CURR:")
print(f"      NULL: {sk_curr_null}")
print(f"      Distinct: {sk_curr_distinct:,}")
print(f"      Min: {sk_curr_min:,}  Max: {sk_curr_max:,}")
print(f"      Registros por cliente: {bronze_row_count / sk_curr_distinct:.2f} (média)")

# SK_ID_BUREAU
sk_bureau_null = df_bureau_bronze.filter(F.col("SK_ID_BUREAU").isNull()).count()
sk_bureau_distinct = df_bureau_bronze.select("SK_ID_BUREAU").distinct().count()
sk_bureau_dups = bronze_row_count - sk_bureau_distinct
sk_bureau_min = df_bureau_bronze.select(F.min("SK_ID_BUREAU")).collect()[0][0]
sk_bureau_max = df_bureau_bronze.select(F.max("SK_ID_BUREAU")).collect()[0][0]
print(f"\n   SK_ID_BUREAU:")
print(f"      NULL: {sk_bureau_null}")
print(f"      Distinct: {sk_bureau_distinct:,}")
print(f"      Duplicatas: {sk_bureau_dups}")
print(f"      Min: {sk_bureau_min:,}  Max: {sk_bureau_max:,}")

# ----------------------------------------------------------------------------
# Distribuição de registros por SK_ID_CURR
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print("DISTRIBUIÇÃO DE REGISTROS POR SK_ID_CURR")
print(f"{'─' * 70}")

per_curr_stats = df_bureau_bronze.groupBy("SK_ID_CURR").count().select("count")
summary = per_curr_stats.summary("min", "max", "mean", "50%").collect()
for s in summary:
    print(f"   {s['summary']:<10}: {s['count']}")

# Top 10 clientes com mais registros
print(f"\n   Top 10 clientes com mais registros de bureau:")
top10 = df_bureau_bronze.groupBy("SK_ID_CURR").count().orderBy(F.desc("count")).limit(10)
display(top10)

# Distribuição por faixas
print(f"\n   Distribuição por faixas de registros:")
bins = [(1, 1), (2, 5), (6, 10), (11, 20), (21, 50), (51, 100), (101, 200)]
for lo, hi in bins:
    cnt = df_bureau_bronze.groupBy("SK_ID_CURR").count().filter(
        (F.col("count") >= lo) & (F.col("count") <= hi)
    ).count()
    print(f"      {lo:>3}-{hi:<3} registros: {cnt:>8,} clientes")

print("\n✅ Perfil e validação de identificadores concluídos!")

In [0]:
# ============================================================================
# CÉLULA 5 — Análise de Duplicidades e Valores Especiais
# ============================================================================
# Analisa duplicidades completas, por SK_ID_BUREAU e valores especiais
# em colunas DAYS_* e monetárias.

print("=" * 70)
print("ANÁLISE DE DUPLICIDADES")
print("=" * 70)

# ----------------------------------------------------------------------------
# Duplicidade completa (linhas totalmente iguais)
# ----------------------------------------------------------------------------
full_dups = bronze_row_count - df_bureau_bronze.dropDuplicates().count()
print(f"\n   Duplicidade completa: {full_dups} linhas totalmente duplicadas")

# ----------------------------------------------------------------------------
# Duplicidade por SK_ID_BUREAU (identificador do registro)
# ----------------------------------------------------------------------------
print(f"   Duplicidade por SK_ID_BUREAU: {sk_bureau_dups} duplicatas")
if sk_bureau_dups == 0:
    print("   → SK_ID_BUREAU é único — nenhum tratamento necessário")
else:
    print("   → ATENÇÃO: SK_ID_BUREAU deveria ser único — investigar duplicatas")

# ----------------------------------------------------------------------------
# Valores especiais em colunas DAYS_*
# ----------------------------------------------------------------------------
print(f"\n{'=' * 70}")
print("VALORES ESPECIAIS — COLUNAS DAYS_*")
print("=" * 70)

days_cols = [c for c in df_bureau_bronze.columns if c.startswith("DAYS_")]
for c in days_cols:
    null_cnt = df_bureau_bronze.filter(F.col(c).isNull()).count()
    pos_cnt = df_bureau_bronze.filter(F.col(c) > 0).count()
    neg_extreme = df_bureau_bronze.filter(F.col(c) < -30000).count()
    print(f"\n   {c}:")
    print(f"      NULL: {null_cnt:,} ({null_cnt/bronze_row_count*100:.2f}%)")
    print(f"      Valores positivos: {pos_cnt:,} ({pos_cnt/bronze_row_count*100:.2f}%)")
    print(f"      Valores < -30000: {neg_extreme:,}")
    if c == "DAYS_CREDIT_ENDDATE" and pos_cnt > 0:
        print(f"      → INTERPRETAÇÃO: Valores positivos indicam data de fim no futuro")
        print(f"        (crédito ainda ativo na data da aplicação). Comportamento esperado.")
        print(f"        Flag FLAG_CREDIT_ACTIVE_ENDDATE será criada para rastreabilidade.")
    if c == "DAYS_CREDIT_UPDATE" and pos_cnt > 0:
        print(f"      → ATENÇÃO: {pos_cnt} registros com DAYS_CREDIT_UPDATE > 0")
        print(f"        (atualização posterior à aplicação). Anomalia potencial.")
        print(f"        Flag FLAG_DAYS_CREDIT_UPDATE_FUTURE será criada.")

# ----------------------------------------------------------------------------
# Valores especiais em colunas monetárias
# ----------------------------------------------------------------------------
print(f"\n{'=' * 70}")
print("VALORES ESPECIAIS — COLUNAS MONETÁRIAS")
print("=" * 70)

amt_cols = [c for c in df_bureau_bronze.columns if c.startswith("AMT_")]
for c in amt_cols:
    null_cnt = df_bureau_bronze.filter(F.col(c).isNull()).count()
    neg_cnt = df_bureau_bronze.filter(F.col(c) < 0).count()
    zero_cnt = df_bureau_bronze.filter(F.col(c) == 0).count()
    print(f"\n   {c}:")
    print(f"      NULL: {null_cnt:,} ({null_cnt/bronze_row_count*100:.2f}%)")
    print(f"      Negativos: {neg_cnt:,} ({neg_cnt/bronze_row_count*100:.2f}%)")
    print(f"      Zeros: {zero_cnt:,} ({zero_cnt/bronze_row_count*100:.2f}%)")
    if neg_cnt > 0:
        print(f"      → WARNING: {neg_cnt} valores negativos. Preservados — podem ter significado próprio.")
        if c == "AMT_CREDIT_SUM_DEBT":
            print(f"        (débito negativo pode indicar saldo credor/overpayment)")
        if c == "AMT_CREDIT_SUM_LIMIT":
            print(f"        (limite negativo pode indicar limite excedido)")

# ----------------------------------------------------------------------------
# CNT_CREDIT_PROLONG — contagem de prolongamentos
# ----------------------------------------------------------------------------
print(f"\n{'=' * 70}")
print("DISTRIBUIÇÃO — CNT_CREDIT_PROLONG")
print("=" * 70)
cnt_dist = df_bureau_bronze.groupBy("CNT_CREDIT_PROLONG").count().orderBy("CNT_CREDIT_PROLONG").collect()
for r in cnt_dist:
    print(f"   {r['CNT_CREDIT_PROLONG']}: {r['count']:,} ({r['count']/bronze_row_count*100:.2f}%)")

print("\n✅ Análise de duplicidades e valores especiais concluída!")

In [0]:
# ============================================================================
# CÉLULA 6 — Integridade Referencial
# ============================================================================
# Valida a relação SK_ID_CURR da bureau contra as tabelas Silver de application.
# Não exclui registros sem correspondência — apenas diagnostica a qualidade.

print("=" * 70)
print("INTEGRIDADE REFERENCIAL — bureau vs application")
print("=" * 70)

bureau_sk = df_bureau_bronze.select("SK_ID_CURR").distinct()
bureau_sk_count = bureau_sk.count()

# ----------------------------------------------------------------------------
# Verificar se tabelas Silver de application existem
# ----------------------------------------------------------------------------
app_tables = {}
for name, tbl in [("application_train", SILVER_APP_TRAIN), ("application_test", SILVER_APP_TEST)]:
    try:
        df_app = spark.table(tbl)
        app_tables[name] = df_app.select("SK_ID_CURR").distinct()
        print(f"   ✅ {tbl}: {app_tables[name].count():,} SK_ID_CURR distintos")
    except Exception:
        print(f"   ⚠️  {tbl}: tabela não encontrada — pulando")

# ----------------------------------------------------------------------------
# Integridade vs application_train
# ----------------------------------------------------------------------------
if "application_train" in app_tables:
    matched_train = bureau_sk.join(app_tables["application_train"], "SK_ID_CURR", "inner").count()
    unmatched_train = bureau_sk_count - matched_train
    print(f"\n📊 Bureau SK_ID_CURR vs application_train:")
    print(f"   Correspondidos: {matched_train:,} ({matched_train/bureau_sk_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_train:,} ({unmatched_train/bureau_sk_count*100:.2f}%)")

# ----------------------------------------------------------------------------
# Integridade vs application_test
# ----------------------------------------------------------------------------
if "application_test" in app_tables:
    matched_test = bureau_sk.join(app_tables["application_test"], "SK_ID_CURR", "inner").count()
    unmatched_test = bureau_sk_count - matched_test
    print(f"\n📊 Bureau SK_ID_CURR vs application_test:")
    print(f"   Correspondidos: {matched_test:,} ({matched_test/bureau_sk_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_test:,} ({unmatched_test/bureau_sk_count*100:.2f}%)")

# ----------------------------------------------------------------------------
# Integridade vs application combinado (train + test)
# ----------------------------------------------------------------------------
if len(app_tables) == 2:
    all_app_sk = app_tables["application_train"].union(app_tables["application_test"]).distinct()
    all_app_count = all_app_sk.count()
    matched_all = bureau_sk.join(all_app_sk, "SK_ID_CURR", "inner").count()
    unmatched_all = bureau_sk_count - matched_all
    print(f"\n📊 Bureau SK_ID_CURR vs application (train + test combinado):")
    print(f"   Total app SK_ID_CURR: {all_app_count:,}")
    print(f"   Correspondidos: {matched_all:,} ({matched_all/bureau_sk_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_all:,} ({unmatched_all/bureau_sk_count*100:.2f}%)")
    if unmatched_all == 0:
        print(f"   → ✅ Todos os SK_ID_CURR da bureau têm correspondência em application")
    else:
        print(f"   → ⚠️  {unmatched_all} SK_ID_CURR sem correspondência (registros preservados)")

print("\n✅ Integridade referencial validada!")

In [0]:
# ============================================================================
# CÉLULA 7 — Funções de Transformação Reutilizáveis
# ============================================================================
# Funções modulares aplicadas na transformação Bronze → Silver.
# Cada função documenta a regra aplicada e retorna o DataFrame transformado.

def remove_bronze_metadata(df, table_name):
    """Remove colunas de metadados da Bronze (_ingestion_timestamp, _source_file).
    Serão substituídas por colunas de controle Silver."""
    cols_to_drop = [c for c in BRONZE_META_COLS if c in df.columns]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        log_transform(table_name, "remove_metadata", f"Removidas colunas Bronze: {cols_to_drop}")
    return df


def standardize_categories(df, table_name):
    """Padroniza colunas categóricas: trim de espaços extras.
    Não altera semântica dos valores — apenas remove espaços à direita/esquerda."""
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    for col_name in string_cols:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

    log_transform(table_name, "standardize_categories",
                  f"Trim aplicado em {len(string_cols)} colunas string")
    print(f"   ✅ Padronização: trim aplicado em {len(string_cols)} colunas string")
    return df


def add_validation_flags(df, table_name, row_count):
    """Adiciona flags de validação para valores potencialmente inválidos.
    Não remove registros — apenas marca anomalias para análise posterior.

    Flags criadas:
    - FLAG_AMT_CREDIT_SUM_INVALID: AMT_CREDIT_SUM é NULL
    - FLAG_AMT_CREDIT_SUM_DEBT_NEGATIVE: AMT_CREDIT_SUM_DEBT < 0 (WARNING — pode ser válido)
    - FLAG_AMT_CREDIT_SUM_LIMIT_NEGATIVE: AMT_CREDIT_SUM_LIMIT < 0 (WARNING — pode ser válido)
    - FLAG_DAYS_CREDIT_UPDATE_FUTURE: DAYS_CREDIT_UPDATE > 0 (anomalia potencial)
    - FLAG_CREDIT_ACTIVE_ENDDATE: DAYS_CREDIT_ENDDATE > 0 (informativo — crédito ativo)
    """
    flags_added = []

    # AMT_CREDIT_SUM — não deveria ser NULL (apenas 13 registros)
    if "AMT_CREDIT_SUM" in df.columns:
        invalid = df.filter(F.col("AMT_CREDIT_SUM").isNull()).count()
        df = df.withColumn("FLAG_AMT_CREDIT_SUM_INVALID",
            F.when(F.col("AMT_CREDIT_SUM").isNull(), 1).otherwise(0))
        flags_added.append(("FLAG_AMT_CREDIT_SUM_INVALID", invalid))

    # AMT_CREDIT_SUM_DEBT — valores negativos (WARNING)
    if "AMT_CREDIT_SUM_DEBT" in df.columns:
        neg = df.filter(F.col("AMT_CREDIT_SUM_DEBT") < 0).count()
        df = df.withColumn("FLAG_AMT_CREDIT_SUM_DEBT_NEGATIVE",
            F.when(F.col("AMT_CREDIT_SUM_DEBT") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_CREDIT_SUM_DEBT_NEGATIVE", neg))

    # AMT_CREDIT_SUM_LIMIT — valores negativos (WARNING)
    if "AMT_CREDIT_SUM_LIMIT" in df.columns:
        neg = df.filter(F.col("AMT_CREDIT_SUM_LIMIT") < 0).count()
        df = df.withColumn("FLAG_AMT_CREDIT_SUM_LIMIT_NEGATIVE",
            F.when(F.col("AMT_CREDIT_SUM_LIMIT") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_CREDIT_SUM_LIMIT_NEGATIVE", neg))

    # DAYS_CREDIT_UPDATE — valores positivos (anomalia: atualização após aplicação)
    if "DAYS_CREDIT_UPDATE" in df.columns:
        future = df.filter(F.col("DAYS_CREDIT_UPDATE") > 0).count()
        df = df.withColumn("FLAG_DAYS_CREDIT_UPDATE_FUTURE",
            F.when(F.col("DAYS_CREDIT_UPDATE") > 0, 1).otherwise(0))
        flags_added.append(("FLAG_DAYS_CREDIT_UPDATE_FUTURE", future))

    # DAYS_CREDIT_ENDDATE — valores positivos (informativo: crédito ativo)
    if "DAYS_CREDIT_ENDDATE" in df.columns:
        active_end = df.filter(F.col("DAYS_CREDIT_ENDDATE") > 0).count()
        df = df.withColumn("FLAG_CREDIT_ACTIVE_ENDDATE",
            F.when(F.col("DAYS_CREDIT_ENDDATE") > 0, 1).otherwise(0))
        flags_added.append(("FLAG_CREDIT_ACTIVE_ENDDATE", active_end))

    for flag_name, affected_count in flags_added:
        log_transform(table_name, "validation_flag",
                      f"{flag_name}: {affected_count} registros marcados",
                      affected_count)
        print(f"   ✅ {flag_name}: {affected_count} registros")

    return df


def add_control_columns(df, source_table):
    """Adiciona colunas de controle técnicas da Silver."""
    df = df.withColumn("silver_processing_timestamp", F.current_timestamp())
    df = df.withColumn("silver_processing_date", F.current_date())
    df = df.withColumn("silver_pipeline_version", F.lit(PIPELINE_VERSION))
    df = df.withColumn("source_table", F.lit(source_table))

    # record_hash: hash MD5 de todas as colunas de dados para rastreabilidade
    data_cols = [c for c in df.columns if c not in [
        "silver_processing_timestamp", "silver_processing_date",
        "silver_pipeline_version", "source_table"
    ]]
    hash_expr = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in data_cols])
    df = df.withColumn("record_hash", F.md5(hash_expr))

    print(f"   ✅ Colunas de controle adicionadas (timestamp, date, version, source, hash)")
    return df


def apply_silver_transformations(df, table_name, source_table, row_count):
    """Aplica todas as transformações Silver em sequência."""
    print(f"\n{'─' * 60}")
    print(f"🔧 Transformando: {table_name}")
    print(f"{'─' * 60}")

    # 1. Remover metadados Bronze
    df = remove_bronze_metadata(df, table_name)

    # 2. Padronizar categorias (trim)
    df = standardize_categories(df, table_name)

    # 3. Adicionar flags de validação
    df = add_validation_flags(df, table_name, row_count)

    # 4. Adicionar colunas de controle
    df = add_control_columns(df, source_table)

    print(f"   ✅ Transformações concluídas para {table_name}")
    return df


print("✅ Funções de transformação definidas!")

In [0]:
# ============================================================================
# CÉLULA 8 — Execução das Transformações
# ============================================================================
# Aplica as transformações Silver na tabela bureau.
# O DataFrame Bronze original não é modificado.

EXEC_START = datetime.now(timezone.utc)

print("=" * 70)
print("TRANSFORMAÇÃO SILVER — bureau")
print("=" * 70)
transform_start = datetime.now(timezone.utc)

df_bureau_silver = apply_silver_transformations(
    df_bureau_bronze, SILVER_TABLE, BRONZE_TABLE, bronze_row_count
)

transform_end = datetime.now(timezone.utc)
transform_duration = (transform_end - transform_start).total_seconds()
silver_row_count = df_bureau_silver.count()
silver_col_count = len(df_bureau_silver.columns)

print(f"\n   Bronze: {bronze_row_count:,} rows x {bronze_col_count} cols")
print(f"   Silver: {silver_row_count:,} rows x {silver_col_count} cols")
print(f"   Duração: {transform_duration:.1f}s")

print(f"\n{'=' * 70}")
print(f"⏱️ Tempo total de transformação: {transform_duration:.1f}s")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 9 — Escrita da Tabela Silver (Delta Lake)
# ============================================================================
# Grava a tabela Silver usando mode("overwrite") com overwriteSchema.
# Isso é seguro porque a Silver é reconstruída a cada execução controlada.
# A Bronze NUNCA é sobrescrita.

print("=" * 70)
print("GRAVAÇÃO DA TABELA SILVER")
print("=" * 70)

print(f"\n📊 Gravando {SILVER_TABLE}...")
write_start = datetime.now(timezone.utc)

df_bureau_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(SILVER_TABLE)

write_end = datetime.now(timezone.utc)
write_duration = (write_end - write_start).total_seconds()
print(f"   ✅ {SILVER_TABLE} gravada em {write_duration:.1f}s")
print(f"      Registros: {silver_row_count:,} | Colunas: {silver_col_count}")

EXEC_END = datetime.now(timezone.utc)
TOTAL_DURATION = (EXEC_END - EXEC_START).total_seconds()

print(f"\n{'=' * 70}")
print("✅ TABELA SILVER GRAVADA COM SUCESSO!")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 10 — Auditoria da Transformação
# ============================================================================
# Cria/atualiza a tabela credit_risk.silver.audit_transformation
# Registra metadados da execução para rastreabilidade histórica (append mode).
from pyspark.sql.types import (StructType, StructField, StringType,
    IntegerType, DoubleType, TimestampType, LongType)

audit_record = {
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "execution_id": EXECUTION_ID,
    "batch_id": BATCH_ID,
    "source_table": BRONZE_TABLE,
    "target_table": SILVER_TABLE,
    "source_row_count": bronze_row_count,
    "target_row_count": silver_row_count,
    "records_inserted": silver_row_count,
    "records_removed": 0,
    "records_changed": silver_row_count,
    "processing_duration_seconds": float(TOTAL_DURATION),
    "pipeline_version": PIPELINE_VERSION,
    "execution_status": "SUCCESS",
    "error_message": "",
}

audit_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("source_row_count", LongType(), True),
    StructField("target_row_count", LongType(), True),
    StructField("records_inserted", LongType(), True),
    StructField("records_removed", IntegerType(), True),
    StructField("records_changed", LongType(), True),
    StructField("processing_duration_seconds", DoubleType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("execution_status", StringType(), True),
    StructField("error_message", StringType(), True),
])

audit_df = spark.createDataFrame([audit_record], schema=audit_schema)

print(f"📊 Persistindo auditoria em {AUDIT_TABLE}...")
audit_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable(AUDIT_TABLE)

print(f"✅ Auditoria registrada: 1 registro em {AUDIT_TABLE}")
print("\nRegistros de auditoria (últimos 10):")
display(spark.table(AUDIT_TABLE).orderBy(F.col("execution_timestamp").desc()).limit(10))

In [0]:
# ============================================================================
# CÉLULA 11 — Data Quality Pós-Transformação (Bronze vs Silver)
# ============================================================================
# Compara métricas de qualidade antes (Bronze) e depois (Silver) para validar
# que as transformações foram aplicadas corretamente.

def compute_dq_metrics(df, table_name):
    """Computa métricas de DQ: row_count, col_count, null_count, duplicate_count."""
    row_count = df.count()
    col_count = len(df.columns)

    # Total de NULLs (soma de todas as colunas)
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) for c in df.columns]
    total_nulls = df.agg(*null_exprs).collect()[0]
    null_sum = sum([total_nulls[i] for i in range(len(df.columns))])

    # Duplicatas por SK_ID_BUREAU
    if "SK_ID_BUREAU" in df.columns:
        dup_count = row_count - df.select("SK_ID_BUREAU").distinct().count()
    else:
        dup_count = 0

    return {
        "table": table_name,
        "row_count": row_count,
        "col_count": col_count,
        "null_count": null_sum,
        "null_percentage": round(null_sum / (row_count * col_count) * 100, 2) if row_count > 0 else 0,
        "duplicate_count": dup_count,
    }

# ----------------------------------------------------------------------------
# Ler tabela Silver recém-criada
# ----------------------------------------------------------------------------
df_silver = spark.table(SILVER_TABLE)

# ----------------------------------------------------------------------------
# Métricas Bronze vs Silver
# ----------------------------------------------------------------------------
print("=" * 70)
print("DATA QUALITY: BRONZE vs SILVER")
print("=" * 70)

bronze_m = compute_dq_metrics(df_bureau_bronze, BRONZE_TABLE)
silver_m = compute_dq_metrics(df_silver, SILVER_TABLE)

print(f"\n📊 bureau")
print(f"{'Métrica':<30} {'Bronze':>15} {'Silver':>15} {'Delta':>15}")
print(f"{'─' * 75}")
for key in ["row_count", "col_count", "null_count", "null_percentage", "duplicate_count"]:
    b = bronze_m[key]
    s = silver_m[key]
    d = s - b
    print(f"{key:<30} {b:>15,} {s:>15,} {d:>+15,}")

# ----------------------------------------------------------------------------
# Verificações específicas
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print("VERIFICAÇÕES ESPECÍFICAS")
print(f"{'─' * 70}")

# Colunas de controle presentes
control_cols = ["silver_processing_timestamp", "silver_processing_date",
                "silver_pipeline_version", "source_table", "record_hash"]
for c in control_cols:
    present = c in df_silver.columns
    print(f"   Coluna {c}: {'✅ presente' if present else '❌ ausente'}")

# Flags de validação
validation_flags = ["FLAG_AMT_CREDIT_SUM_INVALID", "FLAG_AMT_CREDIT_SUM_DEBT_NEGATIVE",
                     "FLAG_AMT_CREDIT_SUM_LIMIT_NEGATIVE", "FLAG_DAYS_CREDIT_UPDATE_FUTURE",
                     "FLAG_CREDIT_ACTIVE_ENDDATE"]
print(f"\n   Flags de validação:")
for flag in validation_flags:
    if flag in df_silver.columns:
        count = df_silver.filter(F.col(flag) == 1).count()
        print(f"      {flag}: {count} registros")

# Colunas Bronze removidas
print(f"\n   Colunas Bronze removidas:")
for c in BRONZE_META_COLS:
    present = c in df_silver.columns
    print(f"      {c}: {'❌ ainda presente' if present else '✅ removida'}")

# SK_ID_BUREAU uniqueness na Silver
silver_sk_dups = silver_row_count - df_silver.select("SK_ID_BUREAU").distinct().count()
print(f"\n   SK_ID_BUREAU duplicatas na Silver: {silver_sk_dups} (esperado: 0)")

print("\n✅ Data Quality pós-transformação concluída!")

In [0]:
# ============================================================================
# CÉLULA 12 — Validação Final e Amostras
# ============================================================================
# Valida que a tabela Silver está correta e coerente com a Bronze.

print("=" * 70)
print("VALIDAÇÃO FINAL — TABELA SILVER")
print("=" * 70)

# ----------------------------------------------------------------------------
# Validação bureau
# ----------------------------------------------------------------------------
print(f"\n📊 {SILVER_TABLE}")
print(f"{'─' * 50}")

# Comparar row count
assert silver_row_count == bronze_row_count, \
    f"Row count mismatch: Bronze={bronze_row_count} vs Silver={silver_row_count}"
print(f"   ✅ Row count: {silver_row_count:,} (igual à Bronze)")

# Comparar SK_ID_BUREAU uniqueness
silver_dups = silver_row_count - df_silver.select("SK_ID_BUREAU").distinct().count()
assert silver_dups == 0, f"Duplicatas encontradas: {silver_dups}"
print(f"   ✅ SK_ID_BUREAU: único (0 duplicatas)")

# Colunas Silver vs Bronze
print(f"   Colunas Bronze: {bronze_col_count}")
print(f"   Colunas Silver: {silver_col_count}")
print(f"   Colunas adicionadas: {silver_col_count - bronze_col_count}")
print(f"     - Removidas: {len([c for c in BRONZE_META_COLS if c in df_bureau_bronze.columns])} (metadados Bronze)")
print(f"     - Adicionadas: 5 flags + 5 colunas controle")

# ----------------------------------------------------------------------------
# Amostra
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print(f"AMOSTRA — {SILVER_TABLE} (primeiras 20 linhas)")
print(f"{'─' * 70}")

sample_cols = [
    "SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "CREDIT_CURRENCY",
    "DAYS_CREDIT", "CREDIT_DAY_OVERDUE", "DAYS_CREDIT_ENDDATE",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE",
    "CREDIT_TYPE", "DAYS_CREDIT_UPDATE", "AMT_ANNUITY",
    "FLAG_AMT_CREDIT_SUM_DEBT_NEGATIVE", "FLAG_CREDIT_ACTIVE_ENDDATE",
    "silver_processing_timestamp", "silver_pipeline_version",
    "source_table", "record_hash"
]
sample_cols = [c for c in sample_cols if c in df_silver.columns]
display(df_silver.select(*sample_cols).limit(20))

# ----------------------------------------------------------------------------
# Estatísticas de SK_ID_CURR e SK_ID_BUREAU na Silver
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print("ESTATÍSTICAS — SK_ID_CURR e SK_ID_BUREAU (Silver)")
print(f"{'─' * 70}")

silver_sk_curr = df_silver.select("SK_ID_CURR").distinct().count()
silver_sk_bureau = df_silver.select("SK_ID_BUREAU").distinct().count()
print(f"   SK_ID_CURR distintos: {silver_sk_curr:,}")
print(f"   SK_ID_BUREAU distintos: {silver_sk_bureau:,}")
print(f"   Registros por cliente (média): {silver_row_count / silver_sk_curr:.2f}")

# Distribuição de CREDIT_ACTIVE na Silver
print(f"\n   Distribuição de CREDIT_ACTIVE (Silver):")
credit_active_dist = df_silver.groupBy("CREDIT_ACTIVE").count().orderBy(F.desc("count")).collect()
for r in credit_active_dist:
    print(f"      {r['CREDIT_ACTIVE']:<20} {r['count']:>12,}")

print("\n✅ Validação final concluída com sucesso!")

In [0]:
# ============================================================================
# CÉLULA 13 — Resumo Final da Execução
# ============================================================================
# Exibe um resumo completo da transformação.

print("=" * 60)
print("SILVER BUREAU - RESUMO")
print("=" * 60)

print(f"\nOrigem:\n  {BRONZE_TABLE}")
print(f"\nDestino:\n  {SILVER_TABLE}")
print(f"\nRegistros Bronze:\n  {bronze_row_count:,}")
print(f"\nRegistros Silver:\n  {silver_row_count:,}")
print(f"\nColunas:\n  Bronze: {bronze_col_count}")
print(f"  Silver: {silver_col_count}")
print(f"\nRegistros removidos:\n  0")
print(f"\nDuplicidades identificadas:\n  Completa: 0")
print(f"  SK_ID_BUREAU: 0")
print(f"\nDuplicidades removidas:\n  0 (não havia duplicidades)")

# Valores inválidos identificados (flags)
print(f"\nValores inválidos identificados:")
for t in TRANSFORMATION_LOG:
    if t["step"] == "validation_flag":
        print(f"  • {t['description']}")

# Integridade referencial
print(f"\nRegistros sem correspondência em application:")
if len(app_tables) == 2:
    all_app_sk = app_tables["application_train"].union(app_tables["application_test"]).distinct()
    unmatched = bureau_sk_count - bureau_sk.join(all_app_sk, "SK_ID_CURR", "inner").count()
    print(f"  {unmatched} (0% — todos correspondem a train+test)")
else:
    print(f"  Verificação não realizada (tabelas Silver de application indisponíveis)")

# Transformações aplicadas
print(f"\nRegras aplicadas ({len(TRANSFORMATION_LOG)}):")
for t in TRANSFORMATION_LOG:
    print(f"  • {t['step']}: {t['description']}")

print(f"\nStatus:\n  SUCCESS")
print(f"\nTempo:\n  {TOTAL_DURATION:.1f} segundos")
print(f"\n⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"\n{'=' * 60}")
print("✅ PIPELINE SILVER BUREAU CONCLUÍDO COM SUCESSO!")
print(f"{'=' * 60}")

## Transformações Aplicadas — Documentação

### 1. Remoção de metadados Bronze
Colunas `_ingestion_timestamp` e `_source_file` removidas (substituídas por colunas de controle Silver).

### 2. Padronização de categorias
- `trim()` aplicado em todas as colunas string (`CREDIT_ACTIVE`, `CREDIT_CURRENCY`, `CREDIT_TYPE`)
- Não há alteração semântica dos valores

### 3. Tratamento de NULLs
- **Categóricos**: Nenhum NULL encontrado nas 3 colunas categóricas — nenhum tratamento necessário
- **Numéricos**: NULLs preservados (imputação estatística é responsabilidade do Feature Engineering/ML)
- Colunas com NULLs significativos:
  - `AMT_ANNUITY` (71.47% — crédito sem anuidade)
  - `AMT_CREDIT_MAX_OVERDUE` (65.51% — sem atraso máximo)
  - `DAYS_ENDDATE_FACT` (36.92% — apenas para créditos encerrados)
  - `AMT_CREDIT_SUM_LIMIT` (34.48% — sem limite de crédito)
  - `AMT_CREDIT_SUM_DEBT` (15.01% — sem dívida)
  - `DAYS_CREDIT_ENDDATE` (6.15% — data de fim desconhecida)
  - `AMT_CREDIT_SUM` (0.00% — apenas 13 registros)

### 4. Valores especiais — DAYS_*

| Coluna | Valores positivos | Interpretação | Tratamento |
|--------|------------------|---------------|------------|
| `DAYS_CREDIT` | 0 | Todos negativos (dias antes da aplicação) | Nenhum |
| `DAYS_CREDIT_ENDDATE` | 602.603 (35.1%) | Data de fim no futuro (crédito ativo) | Flag `FLAG_CREDIT_ACTIVE_ENDDATE` |
| `DAYS_ENDDATE_FACT` | 0 | Todos ≤ 0 (dias desde o fim) | Nenhum |
| `DAYS_CREDIT_UPDATE` | 17 (0.001%) | Atualização após a aplicação (anomalia) | Flag `FLAG_DAYS_CREDIT_UPDATE_FUTURE` |

### 5. Valores especiais — Monetários

| Coluna | Negativos | Interpretação | Tratamento |
|--------|-----------|---------------|------------|
| `AMT_CREDIT_SUM` | 0 | Todos ≥ 0 | Flag para 13 NULLs |
| `AMT_CREDIT_SUM_DEBT` | 8.418 (0.49%) | Possível saldo credor/overpayment | Flag `FLAG_AMT_CREDIT_SUM_DEBT_NEGATIVE` (WARNING) |
| `AMT_CREDIT_SUM_LIMIT` | 351 (0.02%) | Possível limite excedido | Flag `FLAG_AMT_CREDIT_SUM_LIMIT_NEGATIVE` (WARNING) |
| `AMT_CREDIT_SUM_OVERDUE` | 0 | Todos ≥ 0 | Nenhum |
| `AMT_CREDIT_MAX_OVERDUE` | 0 | Todos ≥ 0 | Nenhum |
| `AMT_ANNUITY` | 0 | Todos ≥ 0 | Nenhum |

### 6. Flags de validação
| Flag | Condição | Registros |
|------|---------|----------|
| `FLAG_AMT_CREDIT_SUM_INVALID` | `AMT_CREDIT_SUM` é NULL | 13 |
| `FLAG_AMT_CREDIT_SUM_DEBT_NEGATIVE` | `AMT_CREDIT_SUM_DEBT` < 0 | 8.418 |
| `FLAG_AMT_CREDIT_SUM_LIMIT_NEGATIVE` | `AMT_CREDIT_SUM_LIMIT` < 0 | 351 |
| `FLAG_DAYS_CREDIT_UPDATE_FUTURE` | `DAYS_CREDIT_UPDATE` > 0 | 17 |
| `FLAG_CREDIT_ACTIVE_ENDDATE` | `DAYS_CREDIT_ENDDATE` > 0 | 602.603 |

### 7. Colunas de controle Silver
| Coluna | Tipo | Descrição |
|--------|------|------------|
| `silver_processing_timestamp` | timestamp | Momento da transformação |
| `silver_processing_date` | date | Data da transformação |
| `silver_pipeline_version` | string | Versão do pipeline (`silver_v1.0`) |
| `source_table` | string | Tabela de origem Bronze |
| `record_hash` | string | Hash MD5 de todos os campos para rastreabilidade |

### 8. Identificadores
- `SK_ID_BUREAU` é único (0 duplicatas) — identificador do registro de bureau
- `SK_ID_CURR` tem 305.811 valores distintos — chave estrangeira para application
- Média de 5.61 registros de bureau por cliente (mediana: 4, máximo: 116)

### 9. Integridade referencial
- 100% dos SK_ID_CURR da bureau correspondem a application (train + test combinado)
- 86.16% correspondem apenas a application_train
- 13.84% correspondem apenas a application_test
- Nenhum registro foi removido por falta de correspondência

### 10. Duplicidades
- 0 linhas totalmente duplicadas
- 0 duplicatas de SK_ID_BUREAU
- Nenhuma deduplicação foi necessária